# Querylab step 7: predictions (base vs tuned)

Runtime: GPU (T4). Answers every benchmark prompt with the **base** model and then the **tuned** model (same prompts, greedy decoding, fp16 for both), and writes one predictions file for each.

Before you start: put `bench.jsonl` (from `export_benchmark`) at `MyDrive/querylab/data/bench.jsonl`. The adapter comes from `MyDrive/querylab/run/adapter/` (written by the train notebook). Output goes to `MyDrive/querylab/predictions/`; download both files to your machine.

In [ ]:
# Colab ships TensorFlow, which clashes with the pinned libraries and is not used here.
# After this cell runs once, restart the runtime (Runtime > Restart session) and continue.
!pip uninstall -y -q tensorflow tensorflow-cpu tf-keras keras 2>/dev/null
!pip install -q "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1"

In [ ]:
CONFIG = {
    "base_model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "max_new_tokens": 400,
    "batch_size": 8,
    "drive_dir": "/content/drive/MyDrive/querylab",
}
import os
os.environ["USE_TF"] = "0"  # transformers: never import TensorFlow
os.environ["USE_FLAX"] = "0"
import hashlib, json, torch
from google.colab import drive
drive.mount("/content/drive")
OUT = f"{CONFIG['drive_dir']}/predictions"
os.makedirs(OUT, exist_ok=True)
bench = [json.loads(line) for line in open(f"{CONFIG['drive_dir']}/data/bench.jsonl")]
print(len(bench), "prompts")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"])
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

@torch.no_grad()
def answer_all(model):
    texts = [tok.apply_chat_template([{"role": "system", "content": b["system"]},
                                      {"role": "user", "content": b["prompt"]}],
                                     add_generation_prompt=True, tokenize=False) for b in bench]
    order = sorted(range(len(bench)), key=lambda i: len(texts[i]))
    rows = [None] * len(bench)
    for start in range(0, len(order), CONFIG["batch_size"]):
        idx = order[start:start + CONFIG["batch_size"]]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], do_sample=False,
                             pad_token_id=tok.pad_token_id)
        for row, i in enumerate(idx):
            new = out[row, enc["input_ids"].shape[1]:]
            rows[i] = {"prompt_sha256": bench[i]["prompt_sha256"],
                       "raw_reply": tok.decode(new, skip_special_tokens=True),
                       "input_tokens": int(enc["attention_mask"][row].sum()),
                       "output_tokens": int((new != tok.pad_token_id).sum())}
        print(start + len(idx), "/", len(bench))
    return rows

def write(label, rows, extra):
    path = f"{OUT}/predictions-{label}.jsonl"
    with open(path, "w") as f:
        f.write(json.dumps({"header": {"label": label, "base_model": CONFIG["base_model"], **extra}}) + "\n")
        for row in rows:
            f.write(json.dumps(row) + "\n")
    print("wrote", path)

## Base model

In [ ]:
base = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], torch_dtype=torch.float16, device_map={"": 0})
base.eval()
short = CONFIG["base_model"].split("/")[-1].lower()
write(f"base-{short}", answer_all(base), {"decoding": "greedy"})

## Tuned model (adapter merged into the same fp16 base)

In [ ]:
from peft import PeftModel
adapter = f"{CONFIG['drive_dir']}/run/adapter"
train_config = json.load(open(f"{adapter}/train_config.json"))
tuned = PeftModel.from_pretrained(base, adapter).merge_and_unload()
tuned.eval()
write(f"tuned-{short}-{train_config['adapter_sha256'][:8]}", answer_all(tuned),
      {"decoding": "greedy", "adapter_sha256": train_config["adapter_sha256"],
       "train_config": train_config})